# Optimization

如何找到最佳权重？

$$ w* = argmin_w L(w) $$

IDEA #1: Random Search

IDEA #2: Follow the Slope

一般基于IDEA #2 进行讨论

## 1. 预备知识：计算梯度的方法

### 1.1 analytic gradient
特点：快、准确、可能出错

### 1.2 numerical gradient
特点：慢、不准确、不容易出错

### 1.3 总结
一般而言，用analytic gradient 训练，用numerical gradient check

In [ ]:
import os

# Avoid duplicate OpenMP runtime errors on some local macOS setups.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch.autograd import gradcheck


def loss(w):
    """A tiny toy loss: L(w) = sum(w_i^2)."""
    return torch.sum(w ** 2)


def analytic_gradient(w):
    """Analytic gradient of L(w): dL/dw_i = 2 * w_i."""
    return 2 * w


# gradcheck needs double precision and requires_grad=True.
w = torch.tensor([3.0, -4.0, 1.5], dtype=torch.float64, requires_grad=True)
loss_value = loss(w)
loss_value.backward()

grad_autograd = w.grad.clone()
grad_analytic = analytic_gradient(w.detach())
is_grad_correct = gradcheck(loss, (w.detach().clone().requires_grad_(True),), eps=1e-6, atol=1e-4)

print("w:", w)
print("loss:", loss_value.item())
print("analytic gradient:", grad_analytic)
print("autograd gradient:", grad_autograd)
print("gradcheck passed:", is_grad_correct)


# Gradient descent: repeatedly move weights in the opposite direction of the gradient.
learning_rate = 0.1
w = torch.tensor([3.0, -4.0, 1.5], dtype=torch.float64, requires_grad=True)

for step in range(10):
    loss_value = loss(w)
    loss_value.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        w.grad.zero_()

    print(f"step {step + 1:02d}: loss={loss(w).item():.6f}, w={w.detach()}")


w: tensor([ 3.0000, -4.0000,  1.5000], dtype=torch.float64, requires_grad=True)
loss: 27.25
analytic gradient: tensor([ 6., -8.,  3.], dtype=torch.float64)
autograd gradient: tensor([ 6., -8.,  3.], dtype=torch.float64)
gradcheck passed: True
step 01: loss=17.440000, w=tensor([ 2.4000, -3.2000,  1.2000], dtype=torch.float64)
step 02: loss=11.161600, w=tensor([ 1.9200, -2.5600,  0.9600], dtype=torch.float64)
step 03: loss=7.143424, w=tensor([ 1.5360, -2.0480,  0.7680], dtype=torch.float64)
step 04: loss=4.571791, w=tensor([ 1.2288, -1.6384,  0.6144], dtype=torch.float64)
step 05: loss=2.925946, w=tensor([ 0.9830, -1.3107,  0.4915], dtype=torch.float64)
step 06: loss=1.872606, w=tensor([ 0.7864, -1.0486,  0.3932], dtype=torch.float64)
step 07: loss=1.198468, w=tensor([ 0.6291, -0.8389,  0.3146], dtype=torch.float64)
step 08: loss=0.767019, w=tensor([ 0.5033, -0.6711,  0.2517], dtype=torch.float64)
step 09: loss=0.490892, w=tensor([ 0.4027, -0.5369,  0.2013], dtype=torch.float64)
step 10:

## 2. Gradient Descent
### 2.1 超参数
- 初始权重
- 步数
- 学习率

### 2.2 思想
$$ w_{t+1} = w_t - \alpha \nabla f(w_t) $$
### 2.3 Problem
$$ L(w) = \frac{1}{N} \sum_{i=1}^n L_i(x_i,y_i,w) + \lambda R(w) $$
$$\nabla_w L(w) = \frac{1}{N} \sum_{i=1}^n \nabla_w L_i(x_i,y_i,w) + \nabla _w \lambda R(w)$$
$$N \rightarrow \infty \text{时开销非常大}$$

## 3. Stochastic Gradient Descent （SGD）
### 3.1 超参数
- Batch size
- Data sampling
Note:
- 新增的两个超参并不敏感

### 3.2 思想
sampling，样本估算总体，monte carlo

approximate sum using a mini batch of samples

common: 32/64/128

### 3.3 抽样方法
- shuffle
- batch
- 分批梯度下降
- shuffle
- batch

### 3.4 Problems
- Poor conditioning
- Local minima and saddle points
- approximate



In [2]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch


# Toy problem: learn y = Xw + b from noisy samples.
torch.manual_seed(0)

num_samples = 200
num_features = 2
true_w = torch.tensor([[2.0], [-3.0]])
true_b = torch.tensor([1.0])

X = torch.randn(num_samples, num_features)
y = X @ true_w + true_b + 0.1 * torch.randn(num_samples, 1)


# Parameters to optimize.
w = torch.randn(num_features, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.1
batch_size = 32
num_epochs = 20
# torch.optim.SGD 本身没有 sample_method 这个参数。
# SGD optimizer 只负责根据梯度更新参数，例如 w = w - learning_rate * grad。
# 数据如何抽样、是否 shuffle、是否放回抽样，通常由 DataLoader / Sampler 控制。
# 这里手动写 sampling_method，是为了把 SGD 的抽样思想展示得更清楚。
sampling_method = "shuffle"  # options: "shuffle", "with_replacement"

optimizer = torch.optim.SGD([w, b], lr=learning_rate)


def make_batches(num_samples, batch_size, sampling_method):
    if sampling_method == "shuffle":
        # shuffle: 每个 epoch 先随机打乱数据，再按 batch_size 顺序切分。
        # 这相当于随机不放回抽样，是 mini-batch SGD 里最常见的做法。
        indices = torch.randperm(num_samples)

        for start in range(0, num_samples, batch_size):
            yield indices[start:start + batch_size]

    elif sampling_method == "with_replacement":
        # with_replacement: 每个 batch 都从全体样本中随机抽取。
        # 同一个样本可能在一个 epoch 内被抽到多次，也可能完全没被抽到。
        num_batches = (num_samples + batch_size - 1) // batch_size

        for _ in range(num_batches):
            yield torch.randint(0, num_samples, (batch_size,))

    else:
        raise ValueError(f"Unknown sampling_method: {sampling_method}")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    seen_samples = 0

    for batch_indices in make_batches(num_samples, batch_size, sampling_method):
        X_batch = X[batch_indices]
        y_batch = y[batch_indices]

        y_pred = X_batch @ w + b
        loss = torch.mean((y_pred - y_batch) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(batch_indices)
        seen_samples += len(batch_indices)

    epoch_loss /= seen_samples

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:02d}: loss={epoch_loss:.6f}")


print("learned w:", w.detach().reshape(-1))
print("true w:   ", true_w.reshape(-1))
print("learned b:", b.detach())
print("true b:   ", true_b)


epoch 01: loss=4.994759
epoch 05: loss=0.011988
epoch 10: loss=0.011870
epoch 15: loss=0.011888
epoch 20: loss=0.011814
learned w: tensor([ 2.0017, -3.0112])
true w:    tensor([ 2., -3.])
learned b: tensor([1.0074])
true b:    tensor([1.])


## 4. SGD+Momentum

### 4.1 超参数

- 动量 $\rho$

### 4.2 思想

普通 SGD 只看当下梯度：

$$
w_{t+1}=w_t-\alpha \nabla f(w_t)
$$

Momentum 会额外维护一个 velocity，把过去的更新方向也考虑进来：

$$
v_{t+1}=\rho v_t-\alpha \nabla f(w_t)
$$

$$
w_{t+1}=w_t+v_{t+1}
$$

累计梯度参与决策，而 SGD 只看当下梯度。如果梯度方向长期一致，则走的一步会很大；如果梯度方向来回震荡，momentum 可以抵消一部分震荡。

### 4.3 解决 SGD 的 problems

SGD 的几个典型问题：

- Poor conditioning：不同方向曲率差异大，容易 zig-zag；
- Local minima and saddle points：在平坦区域或 saddle point 附近走得慢；
- approximate：mini-batch gradient 本身有噪声。

Momentum 主要帮助缓解 poor conditioning 和梯度噪声带来的震荡，让更新方向更稳定。


In [3]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch


# Toy problem: learn y = Xw + b from noisy samples.
torch.manual_seed(0)

num_samples = 200
num_features = 2
true_w = torch.tensor([[2.0], [-3.0]])
true_b = torch.tensor([1.0])

X = torch.randn(num_samples, num_features)
y = X @ true_w + true_b + 0.1 * torch.randn(num_samples, 1)

learning_rate = 0.05
batch_size = 32
num_epochs = 20
rho = 0.9  # momentum coefficient, usually 0.9


def train_with_sgd(momentum=0.0):
    # 为了公平比较，普通 SGD 和 SGD+Momentum 从同一个初始点出发。
    torch.manual_seed(1)
    w = torch.randn(num_features, 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)

    # torch.optim.SGD 通过 momentum 参数实现 SGD+Momentum。
    # momentum=0.0 时就是普通 SGD；momentum=rho 时就是带动量的 SGD。
    optimizer = torch.optim.SGD([w, b], lr=learning_rate, momentum=momentum)
    losses = []

    for epoch in range(num_epochs):
        indices = torch.randperm(num_samples)
        epoch_loss = 0.0

        for start in range(0, num_samples, batch_size):
            batch_indices = indices[start:start + batch_size]
            X_batch = X[batch_indices]
            y_batch = y[batch_indices]

            y_pred = X_batch @ w + b
            loss = torch.mean((y_pred - y_batch) ** 2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(batch_indices)

        losses.append(epoch_loss / num_samples)

    return losses, w.detach(), b.detach()


sgd_losses, sgd_w, sgd_b = train_with_sgd(momentum=0.0)
momentum_losses, momentum_w, momentum_b = train_with_sgd(momentum=rho)

print("epoch | SGD loss | Momentum loss")
for epoch in [0, 1, 2, 4, 9, 19]:
    print(f"{epoch + 1:>5} | {sgd_losses[epoch]:>8.6f} | {momentum_losses[epoch]:>13.6f}")

print("\nSGD learned w:", sgd_w.reshape(-1))
print("Momentum learned w:", momentum_w.reshape(-1))
print("true w:", true_w.reshape(-1))
print("SGD learned b:", sgd_b)
print("Momentum learned b:", momentum_b)
print("true b:", true_b)


epoch | SGD loss | Momentum loss
    1 | 8.359269 |      6.049517
    2 | 1.961370 |      3.576297
    3 | 0.521181 |      0.765827
    5 | 0.046125 |      0.221892
   10 | 0.011816 |      0.015280
   20 | 0.011834 |      0.012599

SGD learned w: tensor([ 1.9979, -2.9977])
Momentum learned w: tensor([ 2.0116, -3.0042])
true w: tensor([ 2., -3.])
SGD learned b: tensor([1.0039])
Momentum learned b: tensor([1.0097])
true b: tensor([1.])


## 5. Nesterov Momentum

### 5.1 超参数

- 动量 $\rho$
- learning rate $\alpha$

### 5.2 思想

普通 Momentum 的思想是：先根据当前位置 $x_t$ 的 gradient 更新速度 $v$，再移动参数。

Nesterov Momentum 的区别是：先按照当前 velocity 往前看一步，在 **look ahead point** 上计算 gradient。

$$
v_{t+1}=\rho v_t-\alpha \nabla f(x_t+\rho v_t)
$$

$$
x_{t+1}=x_t+v_{t+1}
$$

这里：

- $\rho v_t$：按照已有 momentum 先往前看一步；
- $x_t+\rho v_t$：look ahead point；
- $\nabla f(x_t+\rho v_t)$：在“预期会到达的位置”计算梯度。

直觉：普通 Momentum 是到了当前位置再看坡度；Nesterov 是先看一下“如果按惯性走下去会到哪里”，然后提前修正方向。

### 5.3 公式变形

原始写法里要算 $\nabla f(x_t+\rho v_t)$，但实际写代码时，backward 通常是对当前保存的变量求梯度。为了更好算，可以令：

$$
\tilde{x}_t=x_t+\rho v_t
$$

则有：

$$
v_{t+1}=\rho v_t-\alpha \nabla f(\tilde{x}_t)
$$

$$
\tilde{x}_{t+1}=\tilde{x}_t-\rho v_t+(1+\rho)v_{t+1}
$$

也可以写成：

$$
\tilde{x}_{t+1}=\tilde{x}_t+v_{t+1}+\rho(v_{t+1}-v_t)
$$

这样代码里维护的 $x$ 实际上是 look ahead 后的 $\tilde{x}$，每次可以直接计算 $\nabla f(x)$。

### 5.4 和 Momentum 的区别

| method | gradient 计算位置 |
|---|---|
| Momentum | $x_t$ |
| Nesterov Momentum | $x_t+\rho v_t$ |


In [4]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch


def f(x):
    # 一个简单的二次函数，最优点在 x=0
    return torch.sum(x ** 2)


rho = 0.9
learning_rate = 0.1
num_steps = 5

# 这里的 x 表示变形后的 look-ahead variable \tilde{x}
x = torch.tensor([3.0], requires_grad=True)
v = torch.zeros_like(x)

print("Nesterov Momentum with reparameterized variable")
for step in range(num_steps):
    loss = f(x)
    loss.backward()

    with torch.no_grad():
        old_v = v.clone()
        grad = x.grad.clone()

        # v_{t+1} = rho * v_t - alpha * grad
        v = rho * v - learning_rate * grad

        # x_{t+1} = x_t - rho * v_t + (1 + rho) * v_{t+1}
        x += -rho * old_v + (1 + rho) * v
        x.grad.zero_()

    print(f"step {step + 1}: loss={f(x).item():.6f}, x={x.item():.6f}, v={v.item():.6f}")


Nesterov Momentum with reparameterized variable
step 1: loss=3.459600, x=1.860000, v=-0.600000
step 2: loss=0.445156, x=0.667200, v=-0.912000
step 3: loss=0.105661, x=-0.325056, v=-0.954240
step 4: loss=0.949590, x=-0.974469, v=-0.793805
step 5: loss=1.555390, x=-1.247153, v=-0.519530


## 6. Adaptive Learning Rates

Adaptive learning rates 的目标是：不同参数维度使用不同的 learning rate。

原因是有些方向变化很快，有些方向变化很慢。如果所有维度都用同一个 learning rate，在 ill-conditioned problem 中可能会出现：

- 某些方向震荡；
- 某些方向走得很慢；
- 整体收敛效率低。

所以 adaptive methods 会根据历史 gradient 自动调整每个维度的步长。

### 6.1 AdaGrad

AdaGrad 会累积每个参数维度上的 squared gradient：

$$
s_t=s_{t-1}+g_t^2
$$

$$
x_{t+1}=x_t-\alpha \frac{g_t}{\sqrt{s_t}+\epsilon}
$$

直觉：如果某个维度历史 gradient 一直很大，说明这个方向变化剧烈，就把它的 effective learning rate 变小；如果某个维度历史 gradient 小，就允许它走得相对大一点。

AdaGrad 可以帮助 overcome ill-conditioned problems。

**Problem:** $s_t$ 会不断累积，分母越来越大，最后 effective learning rate 可能变得太小，导致模型几乎不再更新。

### 6.2 RMSProp

RMSProp 可以理解成 **Leaky AdaGrad**。

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

$$
x_{t+1}=x_t-\alpha \frac{g_t}{\sqrt{s_t}+\epsilon}
$$

RMSProp 不让历史 gradient 无限累积，而是让很久以前的 gradient 慢慢“泄露”掉。因此它缓解了 AdaGrad 后期 learning rate 过小的问题。

### 6.3 Almost Adam

Adam 可以理解成：

$$
\text{Adam}=\text{RMSProp}+\text{Momentum}
$$

为了和前面 Momentum 的记号保持一致，这里把 $v_t$ 留给 momentum / velocity，用 $s_t$ 表示 squared gradient 的 moving average。

令：

$$
g_t=\nabla f(x_t)
$$

Almost Adam 同时维护两个量：

- $v_t$：gradient 的 moving average，类似 momentum；
- $s_t$：squared gradient 的 moving average，类似 RMSProp。

$$
v_t=\beta_1v_{t-1}+(1-\beta_1)g_t
$$

$$
s_t=\beta_2s_{t-1}+(1-\beta_2)g_t^2
$$

然后更新：

$$
x_{t+1}=x_t-\alpha \frac{v_t}{\sqrt{s_t}+\epsilon}
$$

这一步已经有 Adam 的主要想法：一方面用 $v_t$ 平滑更新方向，另一方面用 $s_t$ 给每个参数自适应调整 learning rate。

### 6.4 Adam with Bias Correction

Almost Adam 还有一个问题：初始化时通常令 $v_0=0, s_0=0$，所以训练初期 $v_t, s_t$ 会被 0 拉小，有 initialization bias。

因此真正的 Adam 会加入 bias correction：

$$
\hat{v}_t=\frac{v_t}{1-\beta_1^t}
$$

$$
\hat{s}_t=\frac{s_t}{1-\beta_2^t}
$$

最后更新：

$$
x_{t+1}=x_t-\alpha \frac{\hat{v}_t}{\sqrt{\hat{s}_t}+\epsilon}
$$

所以可以把它拆成两步理解：

1. Almost Adam: RMSProp + Momentum；
2. Adam: Almost Adam + bias correction。

![Adam](adam.png)

![Adam details](adam2.png)

### 6.5 小结

| method | 记录什么 | 主要作用 |
|---|---|---|
| AdaGrad | 历史 squared gradient 总和 $s_t$ | 每个维度自适应步长 |
| RMSProp | squared gradient 的 moving average $s_t$ | 避免 AdaGrad 后期步长太小 |
| Almost Adam | momentum $v_t$ + second moment $s_t$ | Momentum + adaptive learning rate |
| Adam | $\hat{v}_t$ + $\hat{s}_t$ | 在 Almost Adam 上加入 bias correction |


In [5]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch


# ill-conditioned toy problem:
# 第二维曲率大很多，普通 SGD 容易在这个方向上震荡。
def ill_conditioned_loss(x):
    return x[0] ** 2 + 100 * x[1] ** 2


def train_optimizer(optimizer_name, lr=0.05, steps=40):
    x = torch.tensor([5.0, 5.0], requires_grad=True)

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD([x], lr=lr)
    elif optimizer_name == "AdaGrad":
        optimizer = torch.optim.Adagrad([x], lr=lr)
    elif optimizer_name == "RMSProp":
        optimizer = torch.optim.RMSprop([x], lr=lr, alpha=0.9)
    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam([x], lr=lr, betas=(0.9, 0.999))
    else:
        raise ValueError(optimizer_name)

    for _ in range(steps):
        loss = ill_conditioned_loss(x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return ill_conditioned_loss(x).item(), x.detach()


for name in ["SGD", "AdaGrad", "RMSProp", "Adam"]:
    final_loss, final_x = train_optimizer(name)
    print(f"{name:7s} final_loss={final_loss:10.6f}, x={final_x.numpy()}")


# 手写一个 1D Adam，看清楚 Almost Adam 和 bias correction 的区别。
def manual_adam(use_bias_correction, steps=3):
    x = torch.tensor([5.0], requires_grad=True)
    v = torch.zeros_like(x)  # momentum / first moment
    s = torch.zeros_like(x)  # second moment

    alpha = 0.1
    beta1 = 0.9
    beta2 = 0.999
    eps = 1e-8

    print("\nAdam" if use_bias_correction else "\nAlmost Adam")
    for t in range(1, steps + 1):
        loss = torch.sum(x ** 2)
        loss.backward()

        with torch.no_grad():
            g = x.grad.clone()
            v = beta1 * v + (1 - beta1) * g
            s = beta2 * s + (1 - beta2) * (g ** 2)

            if use_bias_correction:
                v_hat = v / (1 - beta1 ** t)
                s_hat = s / (1 - beta2 ** t)
            else:
                v_hat = v
                s_hat = s

            x -= alpha * v_hat / (torch.sqrt(s_hat) + eps)
            x.grad.zero_()

        print(
            f"step={t}, x={x.item():.6f}, "
            f"v={v.item():.6f}, s={s.item():.6f}, "
            f"v_hat={v_hat.item():.6f}, s_hat={s_hat.item():.6f}"
        )


manual_adam(use_bias_correction=False)
manual_adam(use_bias_correction=True)


SGD     final_loss=       nan, x=[0.07390442        nan]
AdaGrad final_loss=1999.222168, x=[4.449076 4.449076]
RMSProp final_loss=751.533691, x=[2.7278063 2.7278063]
Adam    final_loss=969.839294, x=[3.098769 3.098769]

Almost Adam
step=1, x=4.683772, v=1.000000, s=0.100000, v_hat=1.000000, s_hat=0.100000
step=2, x=4.259762, v=1.836754, s=0.187651, v_hat=1.836754, s_hat=0.187651
step=3, x=3.768528, v=2.505031, s=0.260046, v_hat=2.505031, s_hat=0.260046

Adam
step=1, x=4.900000, v=1.000000, s=0.100000, v_hat=10.000000, s_hat=100.000000
step=2, x=4.800058, v=1.880000, s=0.195940, v_hat=9.894737, s_hat=98.019005
step=3, x=4.700213, v=2.652011, s=0.287906, v_hat=9.786020, s_hat=96.064796


## 7. Second-Order Optimization

### 7.1 思想

Second-order optimization 会使用 Hessian，也就是二阶导数信息。

一阶方法只知道 slope；二阶方法还知道 curvature，所以在某些问题上可以走得更准、更快。

### 7.2 Newton Method

对于当前点 $w_0$，Newton method 的更新形式是：

$$
w^* \approx w_0-H_wL(w_0)^{-1}\nabla_wL(w_0)
$$

更一般地写成迭代形式：

$$
w_{t+1}=w_t-H_wL(w_t)^{-1}\nabla_wL(w_t)
$$

其中：

- $\nabla_wL(w_t)$ 是一阶梯度；
- $H_wL(w_t)$ 是 Hessian matrix；
- Hessian 描述 loss surface 的曲率信息。

### 7.3 Problem

在 deep learning 里，second-order optimization 通常不 practical：

1. 参数维度很高，Hessian matrix 非常大；
2. 计算 Hessian 很贵；
3. 求 Hessian inverse 更贵；
4. 非凸情况下 Hessian 可能不是 positive definite，Newton direction 不一定是下降方向。

所以实际深度学习中更常用 SGD、Momentum、RMSProp、Adam 这些 first-order methods。


In [6]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch


def loss_1d(w):
    # 最优点在 w=3 附近，包含二次项和四次项
    return (w - 3) ** 2 + 0.1 * (w - 3) ** 4


w = torch.tensor([8.0], requires_grad=True)

print("Newton method on a 1D toy loss")
for step in range(6):
    loss = loss_1d(w)

    # 一阶导数 gradient
    grad = torch.autograd.grad(loss, w, create_graph=True)[0]

    # 二阶导数 Hessian；1D 情况下 Hessian 就是一个数
    hessian = torch.autograd.grad(grad, w)[0]

    with torch.no_grad():
        w -= grad / hessian

    print(
        f"step {step + 1}: loss={loss_1d(w).item():.8f}, "
        f"w={w.item():.6f}, grad={grad.item():.6f}, hessian={hessian.item():.6f}"
    )

    w.requires_grad_(True)


Newton method on a 1D toy loss
step 1: loss=19.30236816, w=6.125000, grad=60.000000, hessian=32.000000
step 2: loss=4.17002296, w=4.779613, grad=18.457031, hessian=13.718750
step 3: loss=0.64075637, w=3.777332, grad=5.813653, hessian=5.800425
step 4: loss=0.01904937, w=3.137888, grad=1.742544, hessian=2.725094
step 5: loss=0.00000108, w=3.001037, grad=0.276826, hessian=2.022816
step 6: loss=0.00000000, w=3.000000, grad=0.002074, hessian=2.000001


## 8. 小结

从第 5 节开始，这几类方法可以串成一条线：

| Method | 核心思想 | 解决的问题 |
|---|---|---|
| Momentum | 累计过去梯度方向 | SGD 震荡、下降慢 |
| Nesterov Momentum | 在 look ahead point 计算梯度 | 提前修正 momentum 方向 |
| AdaGrad | 每个维度自适应 learning rate | ill-conditioned problem |
| RMSProp | Leaky AdaGrad | AdaGrad 后期 learning rate 太小 |
| Adam | RMSProp + Momentum | 同时利用方向平滑和自适应步长 |
| Second-order | 使用 Hessian 曲率信息 | 理论上更快，但 deep learning 中太贵 |
